In [1]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from read_parquet_from_path import read_parquet_from_blob

c:\Users\pramo\OneDrive\Desktop\Social Media and Mental Health\mental-health-using-emotions\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MODEL_NAME = "duelker/samo-goemotions-deberta-v3-large"

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

Loading weights: 100%|██████████| 394/394 [00:00<00:00, 880.37it/s]


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# Official GoEmotions labels
GOEMOTIONS_LABELS = [
    "admiration",
    "amusement",
    "anger",
    "annoyance",
    "approval",
    "caring",
    "confusion",
    "curiosity",
    "desire",
    "disappointment",
    "disapproval",
    "disgust",
    "embarrassment",
    "excitement",
    "fear",
    "gratitude",
    "grief",
    "joy",
    "love",
    "nervousness",
    "optimism",
    "pride",
    "realization",
    "relief",
    "remorse",
    "sadness",
    "surprise",
    "neutral"
]

# Replace LABEL_0, LABEL_1... with emotion names
emotion_labels = {i: GOEMOTIONS_LABELS[i] for i in range(len(GOEMOTIONS_LABELS))}


def get_emotion_scores(text: str):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    scores = torch.sigmoid(outputs.logits)[0]

    return {
        emotion_labels[i]: float(scores[i])
        for i in range(len(scores))
    }


In [4]:
df = read_parquet_from_blob("Anxiety")

In [ ]:
import logging
import pandas as pd

# Configure logging
logging.basicConfig(
    filename="logs.txt",
    level=logging.INFO,
    format="%(asctime)s | Comment ID: %(message)s"
)

emotion_rows = []

for _, row in df.iterrows():

    if len(emotion_rows) > 5:
        break

    comment_id = row["comment_id"]
    comment = str(row["comment_text"])

    try:
        scores = get_emotion_scores(comment)

        emotion_row = {
            "comment_id": comment_id,
            "comment_text": comment
        }

        for emotion in GOEMOTIONS_LABELS:
            emotion_row[emotion] = scores.get(emotion, 0.0)

        emotion_rows.append(emotion_row)

        logging.info(f"{comment_id} | DONE")

    except Exception as e:
        logging.error(f"{comment_id} | FAILED | {str(e)}")

# Create DataFrame
emotion_df = pd.DataFrame(emotion_rows)